In [26]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

# Load saved PathoPreter probabilities
probs = np.load("100k_probs.npy")
y_true = df["labels"].values

bench_cols = [
    "CADD_raw_rankscore","REVEL_rankscore","AlphaMissense_rankscore",
    "ClinPred_rankscore","PrimateAI_rankscore","MPC_rankscore",
    "BayesDel_addAF_rankscore","EVE_rankscore","ESM1b_rankscore"
]

bench_results = []

# PathoPreter first
bench_results.append((
    "PathoPreter",
    roc_auc_score(y_true, probs),
    average_precision_score(y_true, probs)
))

# DBNSFP models
for c in bench_cols:
    if c in df.columns:
        scores = np.nan_to_num(df[c].to_numpy(), nan=0.0)
        try:
            auc = roc_auc_score(y_true, scores)
            pr  = average_precision_score(y_true, scores)
        except Exception:
            auc, pr = np.nan, np.nan
        bench_results.append((c.replace("_rankscore",""), auc, pr))

# Safe sort (NumPy 2.0 compliant)
bench_results = sorted(
    bench_results,
    key=lambda x: float(x[1]) if x[1] == x[1] else 0.0,
    reverse=True
)

# Print leaderboard
print("\n" + "="*72)
print("🏆 PATHOGENICITY LEADERBOARD — Bottom 100k ClinVar")
print("="*72)
print(f"{'Model':20s} | {'ROC-AUC':8s} | {'PR-AUC':8s}")
print("-"*72)
for name, auc, pr in bench_results:
    print(f"{name:20s} | {auc:8.4f} | {pr:8.4f}")
print("="*72)



🏆 PATHOGENICITY LEADERBOARD — Bottom 100k ClinVar
Model                | ROC-AUC  | PR-AUC  
------------------------------------------------------------------------
ClinPred             |   0.9602 |   0.5921
REVEL                |   0.9554 |   0.5828
AlphaMissense        |   0.9526 |   0.5764
ESM1b                |   0.9440 |   0.5492
BayesDel_addAF       |   0.9355 |   0.6880
PrimateAI            |   0.9298 |   0.4834
MPC                  |   0.9210 |   0.4960
EVE                  |   0.9188 |   0.5107
PathoPreter          |   0.9123 |   0.6204
CADD_raw             |   0.8980 |   0.6566
